# Adult 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Adult 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/adult_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/adult_bttwd.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """读取并展示指定运行情况的结果摘要。"""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} 结果 =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("读取：", dataset_summary)
        display(pd.read_csv(dataset_summary))
    else:
        print("未找到 dataset_summary.csv：", dataset_summary)

    if fold_summary.exists():
        print("读取：", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("未找到 fold_summary.csv：", fold_summary)

    if sample_records.exists():
        print("样本级记录：", sample_records)
    else:
        print("未找到 sample_records.csv：", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/adult_bttwd.yaml
标准输出：
【INFO】【2026-05-17 16:57:04】【数据加载完毕】样本数=32561，特征数=14，正类比例=0.24
【INFO】【2026-05-17 16:57:04】【数据加载】24720 条标签无法映射，未指定负类且未开启 dropna_target，已按 0 处理
【INFO】【2026-05-17 16:57:04】【数据加载】标签列 income 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=32561, 正类比例=24.08%
【INFO】【2026-05-17 16:57:04】【数据集信息】名称=adult，样本数=32561，目标列=income，正类比例=24.08%
【INFO】【2026-05-17 16:57:04】【预处理】连续特征=6个，类别特征=8个
【INFO】【2026-05-17 16:57:04】【预处理】编码后维度=100
【INFO】【2026-05-17 16:57:04】【governance】adult_bttwd 第 1/5 折
【INFO】【2026-05-17 16:57:05】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-17 16:57:05】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-17 16:57:20】【governance】weak bucket resolver：strong=24，weak=45
【INFO】【2026-05-17 16:57:20】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-17 16:57:37】【governance】adult_bttwd 第 2/5 折
【INFO】【2026-05-17 16:57:37】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-17 16:57:37】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-17 16:57:52】【governa

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,average_evidence_count,average_weight_entropy,progressive_closed_count,total_bnd_early_rescue_count,total_bnd_early_rescue_at_leaf_count,total_bnd_early_rescue_at_parent_count,total_root_bnd_after_rescue,avg_bnd_early_rescue_error_rate,avg_bnd_early_rescue_regret,avg_bnd_early_rescue_positive_rate
0,adult_bttwd,0.149756,0.830456,0.727681,0.860907,0.152274,0.81444,0.72023,0.866405,1.0,...,1,0.041432,4216,999,772,227,1096,0.397084,0.448954,0.397287


读取： outputs\governance\full\adult_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,cp_reject_final_error_rate,cp_rejected_pn_count,cp_rejected_pn_ratio,cp_rejected_original_error_rate,cp_rejected_final_error_rate,cp_rejected_final_regret,progressive_update_enabled,average_evidence_count,average_weight_entropy,progressive_closed_count
0,adult_bttwd,1,0.151927,0.827292,0.724200,0.859665,0.156932,0.810392,0.717547,0.867035,...,0.533186,452,0.073664,0.530973,0.533186,0.426549,True,0.234915,0.037912,829
1,adult_bttwd,2,0.146007,0.835858,0.733791,0.863176,0.148403,0.816222,0.721406,0.866093,...,0.556355,417,0.068495,0.556355,0.556355,0.445084,True,0.223894,0.034018,841
2,adult_bttwd,3,0.147405,0.831993,0.731959,0.864251,0.150645,0.810763,0.717165,0.866400,...,0.488834,403,0.066546,0.503722,0.488834,0.391067,True,0.250614,0.041239,859
3,adult_bttwd,4,0.152733,0.827994,0.722388,0.857187,0.153885,0.820118,0.722937,0.864404,...,0.526096,479,0.078589,0.530271,0.526096,0.420877,True,0.270117,0.048407,896
4,adult_bttwd,5,0.150706,0.829146,0.726069,0.860258,0.151505,0.814706,0.722096,0.868090,...,0.563686,369,0.060591,0.563686,0.563686,0.450949,True,0.263974,0.045582,791


样本级记录： outputs\governance\full\adult_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
